# Phase 16L — held-out multilingual evaluation

This notebook **prepares** the three-condition comparison. All GPU-evaluation switches default to `False`. The 98-question translated test set is **provisional**, with limited translation QA and pending human review; the resulting comparison must retain `research_ready: false` and be interpreted as exploratory. No test data or translation is changed here. Run in a fresh Colab runtime with an NVIDIA GPU. The previously measured base-model predictions are reused only after the exact-input/protocol audit passes.


## 1. Verify the assigned GPU

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

smi = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    check=True,
)
print(smi.stdout)
if not smi.stdout.strip():
    raise RuntimeError("No NVIDIA GPU was detected")

## 2. Upload and extract the allowlisted project bundle

On your Mac, first build the Phase 16L bundle using the command in `docs/evaluation/phase16l_colab_evaluation.md`. Upload that `.tar.gz` here. Extraction refuses links and existing destinations.


In [ ]:
from google.colab import files

PROJECT = Path("/content/multilingual-llm-reasoning")
if PROJECT.exists():
    raise FileExistsError(f"Inspect existing project rather than overwriting: {PROJECT}")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one Phase 16L bundle")
archive_path = Path("/content") / next(iter(uploaded))
with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getmembers()
    prefix = "multilingual-llm-reasoning/"
    for member in members:
        name = member.name
        if (
            not name.startswith(prefix)
            or member.issym()
            or member.islnk()
            or not (member.isfile() or member.isdir())
        ):
            raise ValueError(f"Unsafe archive member: {name}")
        if ".." in Path(name).parts:
            raise ValueError(f"Path traversal: {name}")
    archive.extractall("/content", filter="data")
print(PROJECT)

## 3. Install the locked CUDA environment

The project installer creates `/content/multilingual-llm-reasoning/.venv-colab` from the checked-in hash-locked requirements. It does not install packages into the notebook kernel.


In [ ]:
subprocess.run(["bash", "scripts/setup_colab.sh"], cwd=PROJECT, check=True)
PYTHON = PROJECT / ".venv-colab/bin/python"
ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
subprocess.run(
    [
        str(PYTHON),
        "-c",
        (
            "import torch, transformers, peft; "
            "print(torch.__version__, transformers.__version__, peft.__version__, "
            "torch.cuda.is_available()); assert torch.cuda.is_available()"
        ),
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)

## 4. Verify test-grid integrity and baseline reuse

This is read-only. It checks the source manifest and variant-file hash, a 98 × 4 grid with identical canonical IDs and answers, and the saved base-model responses against the exact prompt and decoding contract. It does not claim semantic approval of the translations.


In [ ]:
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "src"))
from multilingual_llm_reasoning.baseline import load_baseline_inputs
from multilingual_llm_reasoning.comparison import load_comparison_config, verify_baseline_reuse
from multilingual_llm_reasoning.translation import LANGUAGES

CONFIG = PROJECT / "configs/phase16l_comparison.yaml"
config = load_comparison_config(CONFIG)
inputs, source_hash, reviewed = load_baseline_inputs(config.baseline)
grid = {(r["original_problem_id"], r["language"]) for r in inputs}
ids = {r["original_problem_id"] for r in inputs}
assert len(ids) == 98 and len(inputs) == len(grid) == 392
assert grid == {(id_, language) for id_ in ids for language in LANGUAGES}
assert reviewed is False and config.allow_provisional is True
baseline_rows, baseline_manifest = verify_baseline_reuse(
    config.baseline_reuse_manifest, config.baseline, inputs, source_hash
)
assert (
    len(baseline_rows) == 392
    and baseline_manifest["complete"]
    and not baseline_manifest["mock_model"]
)
print(
    {
        "canonical_questions": len(ids),
        "languages": list(LANGUAGES),
        "variants": len(inputs),
        "verified_base_responses": len(baseline_rows),
        "research_ready": reviewed,
        "baseline_device": baseline_rows[0]["device"],
        "adapter_device": config.baseline.device,
    }
)

## 5. Mount Drive and import the two frozen Phase 16K adapters

This cell copies only verified final adapters. It checks `completion.json` (75 optimizer steps and SHA-256 for every copied adapter file), refuses existing local adapter targets, then runs the repository's provenance/leakage audit. The Drive training directory is read-only.


In [ ]:
from google.colab import drive

from multilingual_llm_reasoning.comparison_backends import audit_adapter

drive.mount("/content/drive")
TRAINING_DRIVE = Path("/content/drive/MyDrive/multilingual-llm-reasoning/phase16k-full-training-v1")
EVALUATION_DRIVE = Path(
    "/content/drive/MyDrive/multilingual-llm-reasoning/phase16l-final-evaluation-v1"
)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


for drive_name, condition, destination in (
    ("english", "english_lora", config.english_adapter),
    ("multilingual", "multilingual_lora", config.multilingual_adapter),
):
    final = TRAINING_DRIVE / drive_name / "final"
    completion = json.loads((final / "completion.json").read_text())
    if completion.get("status") != "completed" or completion.get("global_step") != 75:
        raise ValueError(f"Incomplete Phase 16K training: {drive_name}")
    required = ("adapter_model.safetensors", "adapter_config.json", "evaluation_provenance.json")
    for name in required:
        source = final / "adapter" / name
        if sha256(source) != completion["artifact_sha256"][f"adapter/{name}"]:
            raise ValueError(f"Drive adapter hash mismatch: {source}")
    if destination.exists():
        raise FileExistsError(f"Local adapter target exists: {destination}")
    shutil.copytree(final / "adapter", destination)
    for name in required:
        if sha256(destination / name) != completion["artifact_sha256"][f"adapter/{name}"]:
            raise ValueError(f"Copied adapter hash mismatch: {destination / name}")
    audit = audit_adapter(destination, condition, config.baseline, ids)
    print(condition, "verified files:", audit["files"])

## 6. Verify CUDA preflight and actual adapter loading

The lightweight preflight checks both adapter provenances, the matched train/validation IDs and training budgets, the test-set separation, and the saved baseline. Optional `VERIFY_ADAPTER_LOAD` loads each adapter on a fresh pinned base model, confirms active PEFT state, then releases it. This can download the pinned base model. It does not generate predictions.


In [ ]:
from scripts.phase16l_evaluation_runner import preflight

_, CONTRACT = preflight()
print(json.dumps(CONTRACT, indent=2))
VERIFY_ADAPTER_LOAD = False
if VERIFY_ADAPTER_LOAD:
    import gc

    import torch

    from multilingual_llm_reasoning.comparison_backends import AdapterBackend

    for condition, adapter_path in (
        ("english_lora", config.english_adapter),
        ("multilingual_lora", config.multilingual_adapter),
    ):
        backend = AdapterBackend(config.baseline, config.baseline.revision, condition, adapter_path)
        print(condition, backend.metadata["adapter_activation"])
        del backend
        gc.collect()
        torch.cuda.empty_cache()

## 7. Baseline condition

The 392 saved, real base-model responses are reused only if the preceding audit passes. The original run used MPS FP16 and the adapter runs use CUDA FP16; the manifest records this device distinction. No base-model inference is launched here.


In [ ]:
print("Base responses eligible for exact-protocol reuse:", len(baseline_rows))
print(
    "Prompt:",
    config.baseline.prompt_version,
    "max_new_tokens:",
    config.baseline.max_new_tokens,
    "seed:",
    config.baseline.seed,
)

## 8. English-only LoRA — 392 variants

Set the switch to `True` deliberately. The runner launches the existing Transformers/PEFT backend in a fresh process and backs up its SQLite progress store to Drive about every 30 seconds. It never silently switches to mock or CPU.


In [ ]:
RUN_ENGLISH_EVAL = False
if RUN_ENGLISH_EVAL:
    subprocess.run(
        [
            str(PYTHON),
            "scripts/phase16l_evaluation_runner.py",
            "--stage",
            "english_lora",
            "--drive-root",
            str(EVALUATION_DRIVE),
            "--confirm-evaluation",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )

## 9. Multilingual LoRA — 392 variants

Run only after the English condition is complete. A separate subprocess loads a fresh base checkpoint and activates only the multilingual adapter.


In [ ]:
RUN_MULTILINGUAL_EVAL = False
if RUN_MULTILINGUAL_EVAL:
    subprocess.run(
        [
            str(PYTHON),
            "scripts/phase16l_evaluation_runner.py",
            "--stage",
            "multilingual_lora",
            "--drive-root",
            str(EVALUATION_DRIVE),
            "--confirm-evaluation",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )

## 10. Finalize the 1,176-row paired comparison

Only after both Drive completion markers exist, enable this cell. It reuses the audited base responses and cached adapter predictions; its strict joins reject missing/duplicate variants. Statistics use a canonical-question-clustered paired bootstrap (5,000 resamples, seed 42). Exact two-sided McNemar tests are per language and unadjusted. Parse failures count incorrect in the full denominator.


In [ ]:
RUN_FINALIZE = False
if RUN_FINALIZE:
    subprocess.run(
        [
            str(PYTHON),
            "scripts/phase16l_evaluation_runner.py",
            "--stage",
            "finalize",
            "--drive-root",
            str(EVALUATION_DRIVE),
            "--confirm-evaluation",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )

## 11. Inspect prediction integrity, metrics, and statistical comparisons

This reads only completed files. It does not invent results when evaluation is pending.


In [ ]:
LOCAL = PROJECT / "outputs/phase16l-comparison-v1"
summary_path = LOCAL / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    assert (
        summary["complete"]
        and summary["expected_predictions"] == 1176
        and summary["research_ready"] is False
    )
    records = [json.loads(line) for line in (LOCAL / "predictions.jsonl").read_text().splitlines()]
    assert (
        len(records)
        == len({(r["condition_id"], r["original_problem_id"], r["language"]) for r in records})
        == 1176
    )
    assert {r["condition_id"] for r in records} == {"base", "english_lora", "multilingual_lora"}
    for row in summary["summary"]:
        print(
            row["condition_id"],
            row["language"],
            f"{row['n_correct']}/{row['n_expected']}",
            "accuracy=",
            row["accuracy"],
            "parse failures=",
            row["n_parse_failures"],
        )
    print("Paired contrasts:", summary["contrasts"])
    print("Exact McNemar by language:", summary["mcnemar"])
else:
    print("Evaluation pending; no result files are claimed")

## 12. Optional offline analysis, charts, and final report

Enable only after finalization. The existing `analyze` and `plot` CLIs read saved predictions. Install the project analysis extra in the notebook kernel (for Matplotlib) separately from the locked CUDA environment, then produce report/figure files and copy them to the new Drive evaluation directory with hashes. Do not tune prompts or adapters using these held-out outcomes.


In [ ]:
RUN_OFFLINE_REPORTS = False
if RUN_OFFLINE_REPORTS:
    if not summary_path.is_file():
        raise FileNotFoundError("Complete comparison required before offline reports")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[analysis]"], cwd=PROJECT, check=True
    )
    from multilingual_llm_reasoning.error_analysis import analyze_results
    from multilingual_llm_reasoning.plotting import plot_metrics

    analysis_dir = LOCAL / "error_analysis"
    figures_dir = LOCAL / "figures"
    analyze_results(
        LOCAL / "predictions.jsonl",
        LOCAL / "selected_test.jsonl",
        analysis_dir,
        seed=42,
        bootstrap_samples=5000,
    )
    plot_metrics(summary_path, figures_dir)
    for source in (analysis_dir, figures_dir):
        target = EVALUATION_DRIVE / "final" / source.name
        if target.exists():
            raise FileExistsError(target)
        shutil.copytree(source, target)
        for path in source.rglob("*"):
            if path.is_file() and sha256(path) != sha256(target / path.relative_to(source)):
                raise ValueError(f"Drive copy failed: {path}")
    print("Verified offline reports:", EVALUATION_DRIVE / "final")

## 13. Recovery after a Colab disconnect

Recreate the same bundle and adapters in a new runtime, rerun the read-only checks, then run the **unfinished** stage with `--resume-from-drive --confirm-evaluation`. The runner validates the unchanged evaluation contract, restores an integrity-checked SQLite backup into an empty local output directory, and skips completed variants. Do not rerun a stage with a completed marker. The matching command is in `docs/evaluation/phase16l_colab_evaluation.md`.
